In [ ]:
#------------------------------------------------------------------------------------------------------#
#
# Code:      CC01_CCAR_B05_model_scoring_01.ipynb
#
# Objective: Step 3: Write a function to use 2016-04 to score new PD model
#
#            Jingru Chen
#            2026-03-22
#
#----------------------------------------------------------------------------------------------------#

# Step 1: Upload libraries

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

run_date="2026-03-22"
my_vintage = 201604

start = datetime.now( ZoneInfo("America/New_York"))

print( start.strftime("%Y-%m-%d %H:%M:%S %Z"))     # 2026-03-17 17:34:58 EDT
print( start.strftime("%Y-%m-%d %I:%M:%S %p %Z"))  # 2026-03-17 05:34:58 PM EDT

2026-03-22 16:57:13 EDT
2026-03-22 04:57:13 PM EDT


In [ ]:
myout= "/content/sample_data"

In [ ]:
pwd

'/content'

In [ ]:
cd /content/sample_data/

/content/sample_data


In [ ]:
ls -ltr

total 84036
-rwxr-xr-x 1 root root      962 Jan  1  2000 README.md*
-rwxr-xr-x 1 root root     1697 Jan  1  2000 anscombe.json*
-rw-r--r-- 1 root root  1706430 Mar 17 17:58 california_housing_train.csv
-rw-r--r-- 1 root root   301141 Mar 17 17:58 california_housing_test.csv
-rw-r--r-- 1 root root 36523880 Mar 17 17:58 mnist_train_small.csv
-rw-r--r-- 1 root root 18289443 Mar 17 17:58 mnist_test.csv
-rw-r--r-- 1 root root     2209 Mar 22 20:14 ccar_pd_model_2026-03-22.pkl
-rw-r--r-- 1 root root 29210174 Mar 22 20:14 CCAR_Mortgage_data_for_model_DEV_20260319_01.csv


In [ ]:
x_list=['original_balance', 'credit_score_orig', 'loan_to_value_orig', 'interest_rate', 'loan_term_months',
        'delta_Unemployment1',
       'delta_Mortgage_rate1', 'delta_House_Price_Index__Level1',
       'delta_Unemployment3', 'delta_Mortgage_rate3',
       'delta_House_Price_Index__Level3', 'delta_Unemployment6',
       'delta_Mortgage_rate6', 'delta_House_Price_Index__Level6',
       'delta_Unemployment12', 'delta_Mortgage_rate12',
       'delta_House_Price_Index__Level12', 'delta_Unemployment24',
       'delta_Mortgage_rate24', 'delta_House_Price_Index__Level24']

x_list_v2=['original_balance', 'credit_score_orig', 'loan_to_value_orig', 'interest_rate',
        'delta_Unemployment1',
       'delta_Unemployment3',
       'delta_Unemployment6',
       'delta_Mortgage_rate6', 'delta_House_Price_Index__Level6',
       'delta_Unemployment12',
       'delta_Unemployment24' ]

# y_list= ['flag_default']

pd_model = 'ccar_pd_model_2026-03-22.pkl'
pd_input_file= "/CCAR_Mortgage_data_for_model_DEV_20260319_01.csv"
pd_vintage = 201604
pd_x_list = x_list_v2
pd_y = 'flag_default'
pd_threshold = 0.6

# Step 2: Write a scoring function & Score vintage=201604

In [ ]:
def my_mortgage_PD_scoring( my_model, my_data, my_vintage, my_x_list, my_y, my_threshold ):

  # Step A:Load the model from your disk/storage
  pd_model = joblib.load( my_model )

  # Step B: Upload data and select a targeted vintage
  df_mortgage= pd.read_csv( myout + my_data )
  df_mortgage= df_mortgage.drop( columns= ['Unnamed: 0'] )

  print( "------------- Step B: Column List of df_mortgage ---------\n", df_mortgage.info() )

  # Step C: Select one targeted vintage for PD model scoring
  df_mortgage_vintage= df_mortgage.loc[df_mortgage.report_yrmo == my_vintage].reset_index(drop=True)

  print( "\n\n------------- Step C-1: Column List of df_mortgage --------:\n", df_mortgage_vintage.report_yrmo.value_counts() )

  print( "------------- Step C-2: Summary Table of df_mortgage ---------\n", pd.crosstab( index= df_mortgage_vintage['report_yrmo'],
                                              columns= df_mortgage_vintage[my_y], margins=True))

  # Step D: Set up X vs y from the selected vintage file
  X = df_mortgage_vintage[ my_x_list ]
  y = df_mortgage_vintage[ my_y ]

  # Step E: Scoring & Confusion matrix
  ### E-1. Generate Predictions
  ### Get probabilities instead of hard predictions
  y_proba = pd_model.predict_proba(X)[:, 1]

  ### E-2: Set a custom threshold based on your portfolio's average default rate
  custom_threshold = my_threshold
  y_pred_new = (y_proba >= custom_threshold).astype(int)


  ### E-2. Calculate Individual Metrics
  accuracy = accuracy_score( y, y_pred_new )
  precision = precision_score( y, y_pred_new )
  recall = recall_score(y, y_pred_new )
  f1 = f1_score( y, y_pred_new )

  ### E-3. Print the Comprehensive Classification Report
  print("\n\n------- E-1: Classification Report -------")
  print(classification_report(y, y_pred_new ))

  ### E-4. Display the Confusion Matrix
  print("------ E-2 Confusion Matrix ------")
  print(confusion_matrix(y, y_pred_new ))


In [ ]:
my_mortgage_PD_scoring( pd_model, pd_input_file, pd_vintage, pd_x_list, pd_y, pd_threshold )



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67957 entries, 0 to 67956
Data columns (total 44 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   loan_id                           67957 non-null  int64  
 1   origination_date                  67957 non-null  object 
 2   report_date                       67957 non-null  object 
 3   num_payments                      67957 non-null  int64  
 4   original_balance                  67957 non-null  float64
 5   current_balance_bk                67957 non-null  float64
 6   EAD                               67957 non-null  float64
 7   credit_score_orig                 67957 non-null  float64
 8   loan_to_value_orig                67957 non-null  float64
 9   interest_rate                     67957 non-null  float64
 10  product_type                      67957 non-null  object 
 11  ever_defaulted                    67957 non-null  bool   
 12  defa

In [ ]:
from datetime import datetime
end = datetime.now(ZoneInfo("America/New_York"))
duration = end - start

print(f"Started:  {start}")
print(f"Finished: {end}")
print(f"\nDuration: {duration}")                    # 0:00:02.351234
print(f"Duration: {duration.total_seconds():.3f} seconds")

Started:  2026-03-22 16:57:13.219435-04:00
Finished: 2026-03-22 16:57:16.423745-04:00

Duration: 0:00:03.204310
Duration: 3.204 seconds
